Import libraries and connect to Neo4j

In [2]:
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv
import pandas as pd
import os

In [6]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

load_dotenv(project_root / ".env")

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USER")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(
    URI,
    auth=(USER, PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


Load extracted CSV files

In [7]:
data_dir = project_root / "data" / "extracted"

entities_df = pd.read_csv(
    data_dir / "entities.csv",
    encoding="utf-8-sig"
)

relationships_df = pd.read_csv(
    data_dir / "relationships.csv",
    encoding="utf-8-sig"
)

print("Entities:", len(entities_df))
print("Relationships:", len(relationships_df))

display(entities_df.head())
display(relationships_df.head())

Entities: 14
Relationships: 13


,id,name_si,label
0,D001,ඇල්ල,Destination
1,DI001,බදුල්ල,District
2,P001,ඌව පළාත,Province
3,A001,පුංචි ශ්‍රී පාදය,Attraction
4,A002,ඇල්ල රොක්,Attraction


,source,relation,target
0,D001,LOCATED_IN,DI001
1,DI001,PART_OF,P001
2,D001,HAS_ATTRACTION,A001
3,D001,HAS_ATTRACTION,A002
4,D001,HAS_ATTRACTION,A003


Create a unique ID constraint
add a common Entity label to every entity, alongside its specific type such as Destination or Attraction

In [8]:
driver.execute_query(
    """
    CREATE CONSTRAINT entity_id_unique IF NOT EXISTS
    FOR (n:Entity)
    REQUIRE n.id IS UNIQUE
    """,
    database_=DATABASE
)

print("Entity ID constraint created!")

Entity ID constraint created!


Insert all 14 entities

In [9]:
allowed_labels = {
    "Destination",
    "District",
    "Province",
    "Attraction",
    "Activity"
}

for row in entities_df.to_dict("records"):
    label = str(row["label"])

    if label not in allowed_labels:
        raise ValueError(f"Unexpected label: {label}")

    query = f"""
    MERGE (n:Entity {{id: $id}})
    SET n:{label}
    SET n.name_si = $name_si
    RETURN n
    """

    driver.execute_query(
        query,
        id=str(row["id"]),
        name_si=str(row["name_si"]),
        database_=DATABASE
    )

print("All entities inserted successfully!")

All entities inserted successfully!


Insert all 13 relationships

In [10]:
allowed_relations = {
    "LOCATED_IN",
    "PART_OF",
    "HAS_ATTRACTION",
    "HAS_ACTIVITY"
}

for row in relationships_df.to_dict("records"):
    relation = str(row["relation"])

    if relation not in allowed_relations:
        raise ValueError(f"Unexpected relationship: {relation}")

    query = f"""
    MATCH (a:Entity {{id: $source}})
    MATCH (b:Entity {{id: $target}})
    MERGE (a)-[r:{relation}]->(b)
    RETURN r
    """

    records, _, _ = driver.execute_query(
        query,
        source=str(row["source"]),
        target=str(row["target"]),
        database_=DATABASE
    )

    if not records:
        raise ValueError(
            f"Could not connect {row['source']} to {row['target']}"
        )

print("All relationships inserted successfully!")

All relationships inserted successfully!


Verify the graph

In [11]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("Total entities:", records[0]["total_nodes"])

records, _, _ = driver.execute_query(
    """
    MATCH (:Entity)-[r]->(:Entity)
    RETURN count(r) AS total_relationships
    """,
    database_=DATABASE
)

print("Total relationships:", records[0]["total_relationships"])

Total entities: 14
Total relationships: 13


graph enrichment

Prepare detailed properties

In [12]:
enrichment_data = [
    {
        "id": "D001",
        "properties": {
            "elevation_m": 1041,
            "climate_si": "සාමාන්‍යයෙන් සිසිල් කඳුකර දේශගුණය",
            "source_section": "1, 2"
        }
    },
    {
        "id": "A001",
        "properties": {
            "elevation_m": 1141,
            "hiking_difficulty": "easy",
            "best_time_si": "උදෑසන සහ සවස් කාලය",
            "source_section": "3"
        }
    },
    {
        "id": "A002",
        "properties": {
            "hiking_difficulty": "challenging",
            "description_si": "පුංචි ශ්‍රී පාදයට සාපේක්ෂව වඩාත් අභියෝගාත්මක කඳු නැගීමකි.",
            "source_section": "4"
        }
    },
    {
        "id": "A004",
        "properties": {
            "height_m": 25,
            "distance_from_ella_km": 6,
            "source_section": "6"
        }
    },
    {
        "id": "A007",
        "properties": {
            "height_m": 220,
            "source_section": "9"
        }
    }
]

print("Entities prepared:", len(enrichment_data))

Entities prepared: 5


Update the existing nodes

In [13]:
for item in enrichment_data:
    records, _, _ = driver.execute_query(
        """
        MATCH (n:Entity {id: $id})
        SET n += $properties
        SET n.source_document = "ella.txt"
        RETURN n.id AS id
        """,
        id=item["id"],
        properties=item["properties"],
        database_=DATABASE
    )

    if not records:
        raise ValueError(
            f"Entity not found: {item['id']}"
        )

print("Node properties updated successfully!")

Node properties updated successfully!


Verify the properties

In [14]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    WHERE n.id IN ["A001", "A002", "A004", "A007"]

    RETURN
        n.name_si AS name,
        n.height_m AS height,
        n.elevation_m AS elevation,
        n.hiking_difficulty AS difficulty

    ORDER BY name
    """,
    database_=DATABASE
)

for record in records:
    print(dict(record))

{'name': 'ඇල්ල රොක්', 'height': None, 'elevation': None, 'difficulty': 'challenging'}
{'name': 'දියලුම දිය ඇල්ල', 'height': 220, 'elevation': None, 'difficulty': None}
{'name': 'පුංචි ශ්\u200dරී පාදය', 'height': None, 'elevation': 1141, 'difficulty': 'easy'}
{'name': 'රාවණා ඇල්ල', 'height': 25, 'elevation': None, 'difficulty': None}


In [15]:
attraction_details = [
    {
        "id": "A001",
        "properties": {
            "name_en": "Little Adam's Peak",
            "aliases": ["පුංචි ශ්‍රී පාදය", "Little Adam's Peak", "Mini Adam's Peak"],
            "description_si": "ඇල්ලේ ජනප්‍රිය කඳු නැගීමේ ස්ථානයකි.",
            "visiting_advice_si": "උදෑසන සහ සවස් කාලය ජනප්‍රියය.",
            "elevation_m": 1141,
            "elevation_approximate": True,
            "source_sections": [3]
        }
    },
    {
        "id": "A002",
        "properties": {
            "name_en": "Ella Rock",
            "aliases": ["ඇල්ල රොක්", "Ella Rock"],
            "description_si": "පුංචි ශ්‍රී පාදයට වඩා අභියෝගාත්මක විය හැකි කඳු නැගීමේ ස්ථානයකි.",
            "visiting_advice_si": "සුදුසු පාවහන් හා ජලය රැගෙන යාම වැදගත් වේ.",
            "source_sections": [4]
        }
    },
    {
        "id": "A003",
        "properties": {
            "name_en": "Nine Arches Bridge",
            "aliases": ["ආරුක්කු නවයේ පාලම", "Nine Arches Bridge"],
            "description_si": "ඇල්ල ආසන්නයේ පිහිටි ප්‍රසිද්ධ දුම්රිය පාලමකි.",
            "popular_for_si": "දුම්රිය නැරඹීම සහ ඡායාරූප ගැනීම",
            "source_sections": [5]
        }
    },
    {
        "id": "A004",
        "properties": {
            "description_si": "ඇල්ලේ සිට කිලෝමීටර් 6ක් පමණ දුරින් පිහිටි දිය ඇල්ලකි.",
            "height_m": 25,
            "height_approximate": True,
            "safety_si": "තෙත් ගල් සහ ශක්තිමත් ජල ධාරා පිළිබඳ සැලකිලිමත් විය යුතුය.",
            "source_sections": [6]
        }
    },
    {
        "id": "A005",
        "properties": {
            "description_si": "රාවණා රජු පිළිබඳ දේශීය පුරාවෘත්ත සමඟ සම්බන්ධ ගුහාවකි.",
            "visiting_advice_si": "ළඟාවීමට පා ගමනක් හෝ කඳු නැගීමක් අවශ්‍ය විය හැකිය.",
            "source_sections": [7]
        }
    },
    {
        "id": "A006",
        "properties": {
            "description_si": "බදුල්ල–බණ්ඩාරවෙල මාර්ගයේ පිහිටි පැරණි බෞද්ධ විහාරස්ථානයකි.",
            "buddha_statue_height_m": 12,
            "statue_height_approximate": True,
            "visiting_advice_si": "සුදුසු ඇඳුමින් ගෞරවාන්විතව හැසිරිය යුතුය.",
            "source_sections": [8]
        }
    },
    {
        "id": "A007",
        "properties": {
            "description_si": "ඇල්ල අවට පුළුල් ප්‍රදේශයේ සංචාරයකදී නැරඹිය හැකි දිය ඇල්ලකි.",
            "height_m": 220,
            "height_approximate": True,
            "source_sections": [9]
        }
    },
    {
        "id": "A008",
        "properties": {
            "description_si": "භූගත පරිසරයක් සහ ජල තටාකයක් සඳහා ප්‍රසිද්ධ ස්ථානයකි.",
            "visiting_advice_si": "පළපුරුදු මාර්ගෝපදේශක සහාය අවශ්‍ය විය හැකිය.",
            "source_sections": [10]
        }
    }
]

for item in attraction_details:
    records, _, _ = driver.execute_query(
        """
        MATCH (n:Entity {id: $id})
        SET n += $properties
        SET n.source_document = "ella.txt"
        RETURN n.id AS id
        """,
        id=item["id"],
        properties=item["properties"],
        database_=DATABASE
    )

    if not records:
        raise ValueError(f"Missing node: {item['id']}")

print("Updated:", len(attraction_details), "attractions")

Updated: 8 attractions


Add transport, accommodation and adventure activities

In [16]:
new_entities = [
    {
        "id": "RS001",
        "label": "RailwayStation",
        "name_si": "ඇල්ල දුම්රිය ස්ථානය",
        "properties": {
            "opened_year": 1918,
            "opened_month": 7,
            "source_sections": [12]
        }
    },
    {
        "id": "T001",
        "label": "TrainService",
        "name_si": "පොඩි මැණිකේ",
        "properties": {
            "train_number": "1005",
            "departure_time": "05:55",
            "arrival_time": "15:14",
            "source_sections": [13]
        }
    },
    {
        "id": "T002",
        "label": "TrainService",
        "name_si": "උඩරට මැණිකේ",
        "properties": {
            "train_number": "1015",
            "departure_time": "08:30",
            "arrival_time": "17:27",
            "source_sections": [13]
        }
    },
    {
        "id": "T003",
        "label": "TrainService",
        "name_si": "දුම්රිය අංක 1007",
        "properties": {
            "train_number": "1007",
            "departure_time": "09:45",
            "arrival_time": "18:22",
            "source_sections": [13]
        }
    },
    {
        "id": "T004",
        "label": "TrainService",
        "name_si": "රාත්‍රී තැපැල් දුම්රිය",
        "properties": {
            "train_number": "1045",
            "departure_time": "20:00",
            "arrival_time": "06:05",
            "arrival_next_day": True,
            "source_sections": [13]
        }
    },
    {
        "id": "H001",
        "label": "Accommodation",
        "name_si": "98 Acres Resort & Spa",
        "properties": {
            "location_si": "Ella–Passara මාර්ගය",
            "source_sections": [17]
        }
    },
    {
        "id": "H002",
        "label": "Accommodation",
        "name_si": "The Secret Ella",
        "properties": {
            "location_si": "Passara Road",
            "source_sections": [17]
        }
    },
    {
        "id": "H003",
        "label": "Accommodation",
        "name_si": "Ella Jungle Resort",
        "properties": {
            "description_si": "ඇල්ල ආසන්නයේ ස්වභාවික පරිසරයක නවාතැන් පහසුකම් සපයයි.",
            "source_sections": [17]
        }
    },
    {
        "id": "H004",
        "label": "Accommodation",
        "name_si": "La Ella Breeze",
        "properties": {
            "facilities_si": ["අවන්හල", "බාර්"],
            "source_sections": [17]
        }
    },
    {
        "id": "AC004",
        "label": "Activity",
        "name_si": "Zipline",
        "properties": {"source_sections": [11]}
    },
    {
        "id": "AC005",
        "label": "Activity",
        "name_si": "ATV රථ ධාවනය",
        "properties": {"source_sections": [11]}
    }
]

In [17]:
allowed_labels = {
    "RailwayStation",
    "TrainService",
    "Accommodation",
    "Activity"
}

for item in new_entities:
    label = item["label"]

    if label not in allowed_labels:
        raise ValueError(f"Invalid label: {label}")

    query = f"""
    MERGE (n:Entity {{id: $id}})
    SET n:{label}
    SET n.name_si = $name
    SET n += $properties
    SET n.source_document = "ella.txt"
    RETURN n.id AS id
    """

    driver.execute_query(
        query,
        id=item["id"],
        name=item["name_si"],
        properties=item["properties"],
        database_=DATABASE
    )

print("New nodes inserted:", len(new_entities))

New nodes inserted: 11


In [18]:
new_relationships = [
    # Attractions and activities
    ("A001", "SUPPORTS_ACTIVITY", "AC001"),
    ("A002", "SUPPORTS_ACTIVITY", "AC001"),
    ("A003", "SUPPORTS_ACTIVITY", "AC003"),

    # Destination and activities
    ("D001", "HAS_ACTIVITY", "AC004"),
    ("D001", "HAS_ACTIVITY", "AC005"),

    # Railway station
    ("D001", "HAS_RAILWAY_STATION", "RS001"),

    # Train services arriving at Ella
    ("T001", "ARRIVES_AT", "RS001"),
    ("T002", "ARRIVES_AT", "RS001"),
    ("T003", "ARRIVES_AT", "RS001"),
    ("T004", "ARRIVES_AT", "RS001"),

    # Accommodation
    ("D001", "HAS_ACCOMMODATION", "H001"),
    ("D001", "HAS_ACCOMMODATION", "H002"),
    ("D001", "HAS_ACCOMMODATION", "H003"),
    ("D001", "HAS_ACCOMMODATION", "H004")
]

allowed_relations = {
    "SUPPORTS_ACTIVITY",
    "HAS_ACTIVITY",
    "HAS_RAILWAY_STATION",
    "ARRIVES_AT",
    "HAS_ACCOMMODATION"
}

for source, relation, target in new_relationships:
    if relation not in allowed_relations:
        raise ValueError(f"Invalid relationship: {relation}")

    query = f"""
    MATCH (a:Entity {{id: $source}})
    MATCH (b:Entity {{id: $target}})
    MERGE (a)-[r:{relation}]->(b)
    SET r.source_document = "ella.txt"
    RETURN r
    """

    records, _, _ = driver.execute_query(
        query,
        source=source,
        target=target,
        database_=DATABASE
    )

    if not records:
        raise ValueError(
            f"Missing nodes for {source} -> {target}"
        )

print("Relationships processed:", len(new_relationships))

Relationships processed: 14


Verify expanded Knowledge Graph

In [19]:
records, _, _ = driver.execute_query(
    """
    MATCH (n:Entity)
    WITH count(n) AS node_count
    MATCH (:Entity)-[r]->(:Entity)
    RETURN node_count, count(r) AS relationship_count
    """,
    database_=DATABASE
)

print(dict(records[0]))

{'node_count': 25, 'relationship_count': 27}


Add source evidence to the Knowledge Graph

In [20]:
source_evidence = [
    {
        "source": "D001",
        "relation": "LOCATED_IN",
        "target": "DI001",
        "section": 1,
        "evidence": "ඇල්ල ශ්‍රී ලංකාවේ ඌව පළාතේ බදුල්ල දිස්ත්‍රික්කයේ පිහිටි කුඩා කඳුකර නගරයකි."
    },
    {
        "source": "D001",
        "relation": "HAS_ATTRACTION",
        "target": "A001",
        "section": 3,
        "evidence": "පුංචි ශ්‍රී පාදය, Little Adam's Peak හෝ Mini Adam's Peak ලෙසද හැඳින්වෙන අතර ඇල්ලේ ජනප්‍රියම සංචාරක ස්ථාන අතරින් එකකි."
    },
    {
        "source": "D001",
        "relation": "HAS_ATTRACTION",
        "target": "A003",
        "section": 5,
        "evidence": "Demodara Nine Arch Bridge හෙවත් ආරුක්කු නවයේ පාලම ඇල්ල ආසන්නයේ පිහිටි ප්‍රසිද්ධම සංචාරක සලකුණු අතරින් එකකි."
    },
    {
        "source": "D001",
        "relation": "HAS_ATTRACTION",
        "target": "A004",
        "section": 6,
        "evidence": "රාවණා ඇල්ල ඇල්ල නගරයේ සිට කිලෝමීටර් 6ක් පමණ දුරින් පිහිටි ජනප්‍රිය දිය ඇල්ලකි."
    }
]

print("Evidence records:", len(source_evidence))

Evidence records: 4


In [21]:
allowed_relations = {"LOCATED_IN", "HAS_ATTRACTION"}

for item in source_evidence:
    relation = item["relation"]

    if relation not in allowed_relations:
        raise ValueError("Unexpected relationship type")

    query = f"""
    MATCH (a:Entity {{id: $source}})
          -[r:{relation}]->
          (b:Entity {{id: $target}})
    SET r.source_document = "ella.txt",
        r.source_section = $section,
        r.source_evidence = $evidence
    RETURN r
    """

    records, _, _ = driver.execute_query(
        query,
        source=item["source"],
        target=item["target"],
        section=item["section"],
        evidence=item["evidence"],
        database_=DATABASE
    )

    if not records:
        raise ValueError(f"Relationship missing: {item}")

print("Source evidence added successfully!")

Source evidence added successfully!


In [22]:
records, _, _ = driver.execute_query(
    """
    MATCH (e:Destination {id: "D001"})
          -[r:HAS_ATTRACTION]->
          (a:Attraction)
    WHERE r.source_evidence IS NOT NULL

    RETURN
        a.name_si AS attraction,
        r.source_section AS section,
        r.source_evidence AS evidence

    ORDER BY section
    """,
    database_=DATABASE
)

for record in records:
    print("\nAttraction:", record["attraction"])
    print("Section:", record["section"])
    print("Evidence:", record["evidence"])


Attraction: පුංචි ශ්‍රී පාදය
Section: 3
Evidence: පුංචි ශ්‍රී පාදය, Little Adam's Peak හෝ Mini Adam's Peak ලෙසද හැඳින්වෙන අතර ඇල්ලේ ජනප්‍රියම සංචාරක ස්ථාන අතරින් එකකි.

Attraction: ආරුක්කු නවයේ පාලම
Section: 5
Evidence: Demodara Nine Arch Bridge හෙවත් ආරුක්කු නවයේ පාලම ඇල්ල ආසන්නයේ පිහිටි ප්‍රසිද්ධම සංචාරක සලකුණු අතරින් එකකි.

Attraction: රාවණා ඇල්ල
Section: 6
Evidence: රාවණා ඇල්ල ඇල්ල නගරයේ සිට කිලෝමීටර් 6ක් පමණ දුරින් පිහිටි ජනප්‍රිය දිය ඇල්ලකි.
